# Planet Order Creation (Nepal Landslides)

Creates Planet orders for after images, and optionally before images.
Tracks all order attempts in Hugging Face at `raw_images/order_log.csv` to prevent duplicates.

In [ ]:
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

# ----------------------------
# User configuration
# ----------------------------
INPUT_CSV = '/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv'
START_IDX = 0   # row-index start (inclusive)
END_IDX = 0     # row-index end (exclusive); 0 means all rows
ORDER_BEFORE = False
REORDER_EXISTING = False

MAX_AOI_DEG = 0.1
CLOUD_MAX = 0.05              # <5%
PRE_DAYS = 180                # up to 6 months before
POST_DAYS = 30                # up to 1 month after
MAX_SCENES_PER_ORDER = 12     # allows multi-scene mosaic downstream

ORDERS_URL = 'https://api.planet.com/compute/ops/orders/v2'
DATA_URL = 'https://api.planet.com/data/v1'
ITEM_TYPE = 'PSScene'
BUNDLE_TYPE = 'analytic_sr_udm2'

HF_REPO_ID = 'sasudo2/landslides'
HF_REPO_TYPE = 'dataset'
HF_REVISION = 'main'
ORDER_LOG_PATH = 'raw_images/order_log.csv'
LOCAL_ORDER_LOG = '/kaggle/working/order_log.csv'

In [ ]:
def clamp_aoi(min_lon, min_lat, max_lon, max_lat, max_deg=MAX_AOI_DEG):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= max_deg and lat_span <= max_deg:
        return float(min_lon), float(min_lat), float(max_lon), float(max_lat)
    cx = (min_lon + max_lon) / 2.0
    cy = (min_lat + max_lat) / 2.0
    half = max_deg / 2.0
    return float(cx - half), float(cy - half), float(cx + half), float(cy + half)

def polygon_from_bbox(min_lon, min_lat, max_lon, max_lat):
    return {
        'type': 'Polygon',
        'coordinates': [[[min_lon, min_lat], [max_lon, min_lat], [max_lon, max_lat], [min_lon, max_lat], [min_lon, min_lat]]],
    }

def make_planet_session(api_key):
    s = requests.Session()
    s.auth = (api_key, '')
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
    )
    s.mount('https://', HTTPAdapter(max_retries=retries))
    return s

def request_or_raise(resp):
    if not resp.ok:
        try:
            detail = resp.json()
        except Exception:
            detail = resp.text
        raise requests.HTTPError(f'{resp.status_code} {resp.reason} - {detail}', response=resp)

def search_scenes(session, aoi_geom, gte_ts, lte_ts):
    payload = {
        'item_types': [ITEM_TYPE],
        'filter': {
            'type': 'AndFilter',
            'config': [
                {'type': 'GeometryFilter', 'field_name': 'geometry', 'config': aoi_geom},
                {'type': 'DateRangeFilter', 'field_name': 'acquired', 'config': {'gte': gte_ts, 'lte': lte_ts}},
                {'type': 'RangeFilter', 'field_name': 'cloud_cover', 'config': {'lte': CLOUD_MAX}},
            ],
        },
    }
    r = session.post(f'{DATA_URL}/quick-search', json=payload, timeout=60)
    request_or_raise(r)
    return r.json().get('features', [])

def sort_features(features, incident_date):
    def key_fn(f):
        acquired = pd.to_datetime(f['properties']['acquired'])
        cloud = float(f['properties'].get('cloud_cover', 1.0) or 1.0)
        return (abs((acquired - incident_date).total_seconds()), cloud)
    return sorted(features, key=key_fn)

def submit_order(session, order_name, item_ids, aoi_geom):
    payload = {
        'name': order_name,
        'products': [{'item_ids': item_ids, 'item_type': ITEM_TYPE, 'product_bundle': BUNDLE_TYPE}],
        'tools': [{'type': 'clip', 'parameters': {'aoi': aoi_geom}}],
    }
    r = session.post(ORDERS_URL, json=payload, timeout=60)
    request_or_raise(r)
    return r.json()

In [ ]:
# Auth + dataframe setup
secrets = UserSecretsClient()
planet_api_key = secrets.get_secret('planet_api_key')
hf_token = secrets.get_secret('huggingface_token')

planet = make_planet_session(planet_api_key)
probe = planet.get(ORDERS_URL, timeout=60)
request_or_raise(probe)
print('Planet auth OK')

hf_api = HfApi(token=hf_token)
hf_api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)
print('Hugging Face dataset ready')

df = pd.read_csv(INPUT_CSV)
if START_IDX == 0 and END_IDX == 0:
    df_sel = df.copy()
else:
    df_sel = df.iloc[START_IDX:END_IDX].copy()
df_sel['incident_on'] = pd.to_datetime(df_sel['incident_on'], dayfirst=True)
df_sel = df_sel.reset_index().rename(columns={'index': 'row_index'})
print(f'Incidents selected: {len(df_sel)}')

# Load prior order log from HF if present
try:
    downloaded = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        revision=HF_REVISION,
        filename=ORDER_LOG_PATH,
        token=hf_token,
    )
    prior_log = pd.read_csv(downloaded)
    print(f'Loaded existing order log rows: {len(prior_log)}')
except Exception:
    prior_log = pd.DataFrame()
    print('No existing order log found; starting fresh')

# Only dedupe on attempts that actually resulted in a live order.
# 'failed' attempts (transient errors, no scenes found yet, etc.) must be retried on the next run.
NON_BLOCKING_STATES = {'failed'}
existing_keys = set()
if not prior_log.empty:
    for _, r in prior_log.iterrows():
        if str(r.get('order_state', '')) in NON_BLOCKING_STATES:
            continue
        key = (int(r['incident_id']), str(r['order_type']))
        existing_keys.add(key)

In [ ]:
records = []
created = skipped = failed = 0

for _, row in df_sel.iterrows():
    inc_id = int(row['id'])
    incident_date = row['incident_on']
    min_lon, min_lat, max_lon, max_lat = clamp_aoi(row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    aoi = polygon_from_bbox(min_lon, min_lat, max_lon, max_lat)

    for order_type in (['after'] + (['before'] if ORDER_BEFORE else [])):
        key = (inc_id, order_type)
        if key in existing_keys and not REORDER_EXISTING:
            skipped += 1
            records.append({
                'incident_id': inc_id, 'title': row['title'], 'incident_on': str(row['incident_on'].date()),
                'min_lon': min_lon, 'min_lat': min_lat, 'max_lon': max_lon, 'max_lat': max_lat,
                'row_index': int(row['row_index']), 'order_type': order_type, 'before_enabled': ORDER_BEFORE,
                'order_name': f'incident_{inc_id}_planet_{order_type}', 'order_id': '', 'order_state': 'skipped_existing',
                'scene_count': 0, 'created_at': datetime.now(timezone.utc).isoformat(), 'error': ''
            })
            continue

        if order_type == 'before':
            start = (incident_date - pd.DateOffset(days=PRE_DAYS)).strftime('%Y-%m-%dT00:00:00Z')
            end = (incident_date - pd.DateOffset(days=1)).strftime('%Y-%m-%dT23:59:59Z')
        else:
            start = (incident_date + pd.DateOffset(days=1)).strftime('%Y-%m-%dT00:00:00Z')
            end = (incident_date + pd.DateOffset(days=POST_DAYS)).strftime('%Y-%m-%dT23:59:59Z')

        order_name = f'incident_{inc_id}_planet_{order_type}'
        err = ''
        order_id = ''
        order_state = 'failed'
        scene_count = 0

        try:
            feats = search_scenes(planet, aoi, start, end)
            if len(feats) == 0:
                raise ValueError('No scenes found in date/cloud window')

            ranked = sort_features(feats, incident_date)
            chosen = ranked[:MAX_SCENES_PER_ORDER]
            item_ids = [f['id'] for f in chosen]
            scene_count = len(item_ids)

            order_json = submit_order(planet, order_name, item_ids, aoi)
            order_id = order_json.get('id', '')
            order_state = order_json.get('state', 'queued')
            created += 1
            print(f'[OK] {order_name}: {scene_count} scenes, state={order_state}')

            existing_keys.add(key)
            time.sleep(0.4)
        except Exception as e:
            failed += 1
            err = str(e)
            print(f'[FAIL] {order_name}: {err}')

        records.append({
            'incident_id': inc_id, 'title': row['title'], 'incident_on': str(row['incident_on'].date()),
            'min_lon': min_lon, 'min_lat': min_lat, 'max_lon': max_lon, 'max_lat': max_lat,
            'row_index': int(row['row_index']), 'order_type': order_type, 'before_enabled': ORDER_BEFORE,
            'order_name': order_name, 'order_id': order_id, 'order_state': order_state,
            'scene_count': scene_count, 'created_at': datetime.now(timezone.utc).isoformat(), 'error': err
        })

run_log = pd.DataFrame.from_records(records)
if prior_log.empty:
    merged = run_log
else:
    merged = pd.concat([prior_log, run_log], ignore_index=True)

merged.to_csv(LOCAL_ORDER_LOG, index=False)
hf_api.upload_file(
    path_or_fileobj=LOCAL_ORDER_LOG,
    path_in_repo=ORDER_LOG_PATH,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    revision=HF_REVISION,
)

print('---')
print(f'Created: {created}')
print(f'Skipped existing: {skipped}')
print(f'Failed: {failed}')
print(f'Order log uploaded to: {ORDER_LOG_PATH}')